In [1]:
import os
import yaml

from typing import TypedDict, Annotated, List, Tuple, Literal, Union
from enum import Enum
from pydantic import BaseModel, Field
import operator

from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent / 'code' / 'tool'))
from rag_tool import create_rag_tool
from text2sql_tool import text2sql_workflow

from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.runnables import RunnableConfig


from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime

from langgraph.types import interrupt, Command
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.store.postgres import PostgresStore

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
def load_prompt(yaml_file):
    with open(yaml_file, "r", encoding="utf-8") as file:
        return yaml.safe_load(file)

In [3]:
# db_url = f"postgresql://postgres:{os.getenv('POSTGRES_PASSWORD')}@{os.getenv('POSTGRES_HOST')}:{os.getenv('POSTGRES_PORT')}/{os.getenv('POSTGRES_NAME')}"

# saver = PostgresSaver.from_conn_string(db_url)
# store = PostgresStore.from_conn_string(db_url)


from langgraph.checkpoint.memory import InMemorySaver

inmemory_saver = InMemorySaver()

## State

In [4]:
class MyAgentState(TypedDict):
    input: Annotated[str, "User's input"]
    
    intent_category: Annotated[str, "Intent category"]
    rewritten_query: Annotated[str, "Rewritten query"]

    chat_history: Annotated[List[BaseMessage], operator.add]

    # Plan and Execute fields
    plan: Annotated[List[str], "Current plan"]
    past_steps: Annotated[List[Tuple], operator.add]
    
    response: Annotated[str, "Final response"]

## 1. Intent Analyze
* 사용자의 input과 history를 고려해 rewritten query와 intent category를 출력
* intent category 종류
    * COMPLEX: 다단계 처리 또는 멀티툴 사용 요구 질의(plan and execute sub-graph로 처리)
    * SIMPLE: 단일 응답으로 해결 가능한 간단 질의
    * INAPPROPRIATE: 서비스 범위 외 또는 부적절 질의
* dynamic prompting: history 유무에 따른 프롬프트 적용

In [5]:
class Category(str, Enum):
    """Defines the query processing categories."""
    COMPLEX = "COMPLEX"
    SIMPLE = "SIMPLE"
    INAPPROPRIATE = "INAPPROPRIATE"

class Intent(BaseModel):
    """Schema containing the user query’s intent, processing category, and rewritten query."""
    category: Category = Field(
        description="Query processing category. Must be one of 'COMPLEX', 'SIMPLE', or 'INAPPROPRIATE'."
    )
    query_rewrite: str = Field(
        description="A rewritten version of the original query to make it easier for downstream Agents to process. If the category is INAPPROPRIATE, contains a rejection message."
    )

parser = PydanticOutputParser(pydantic_object=Intent)

In [ ]:
with_history_prompt = load_prompt('../prompt/intent_analyze/with_history_20251127_01.yaml')
without_history_prompt = load_prompt('../prompt/intent_analyze/without_history_20251127_01.yaml')

with_history_prompt = PromptTemplate(
    template=with_history_prompt['template'],
    input_variables=with_history_prompt['input_variables']
)

# without_history_prompt = PromptTemplate(
#     template=without_history_prompt['template'],
#     input_variables=without_history_prompt['input_variables']
# )

with_history_prompt = with_history_prompt.partial(format=parser.get_format_instructions())
# without_history_prompt = without_history_prompt.partial(format=parser.get_format_instructions())

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

with_chain = with_history_prompt | llm
# without_chain = without_history_prompt | llm

In [35]:
def intentAnalyze(state: MyAgentState, config: RunnableConfig):

    # if state['chat_history'] == []:
    #     result = without_chain.invoke({'user_query': state['input']})
    # else:
    #     result = with_chain.invoke({'user_query': state['input'],
    #                                 'chat_history': state['chat_history']})

    configurable = config.get("configurable", {})
    thread_id = configurable.get("thread_id")
    user_id = configurable.get("user_id")

    result = with_chain.invoke({'user_query': state['input'],
                                'chat_history': state['chat_history'],
                                'user_id': user_id})

    structed_output = parser.parse(result.content)
    return {'intent_category': structed_output.category.value,
            'rewritten_query': structed_output.query_rewrite}

In [36]:
def should_continue_intent(state: MyAgentState) -> Literal['COMPLEX', 'SIMPLE', 'INAPPROPRIATE']:
    return state.get('intent_category', 'SIMPLE')

## 2. Plan And Execute 기반의 sub-graph
* 우선 장기 메모리는 적용하지 않음

In [37]:
# PlanAndExecuteState는 이제 MyAgentState에 통합됨
# class PlanAndExecuteState(TypedDict):
#     rewritten_query: Annotated[str, "Rewritten query"]
#     
#     plan: Annotated[List[str], "Current plan"]
#     past_steps: Annotated[List[Tuple], operator.add]
#     response: Annotated[str, "Final response"]

### 2-1. Plan

In [38]:
class Plan(BaseModel):
    """Sorted steps to execute the plan"""

    steps: Annotated[List[str], "Different steps to follow, should be in sorted order"]

In [39]:
planner_prompt = load_prompt('../prompt/plan_and_execute/plan_20251130_01.yaml')
planner_prompt = PromptTemplate(
    template=planner_prompt['template'],
    input_variables=planner_prompt['input_variables']
)

planner = planner_prompt | ChatOpenAI(model='gpt-4o-mini', temperature=0).with_structured_output(Plan)

In [40]:
def plan_step(state: MyAgentState):
    plan = planner.invoke({"rewritten_query": [("user", state["rewritten_query"])]})
    return {"plan": plan.steps}

### 2-2. Re-Plan

In [41]:
class Response(BaseModel):
    """Response to user."""

    response: str


class Act(BaseModel):
    """Action to perform."""

    # 수행할 작업: "Response", "Plan". 사용자에게 응답할 경우 Response 사용, 추가 도구 사용이 필요할 경우 Plan 사용
    action: Union[Response, Plan] = Field(
        description="Action to perform. If you want to respond to user, use Response. "
        "If you need to further use tools to get the answer, use Plan."
    )

In [42]:
replan_prompt = load_prompt('../prompt/plan_and_execute/replan_20251130_02.yaml')
replan_prompt = PromptTemplate(
    template=replan_prompt['template'],
    input_variables=replan_prompt['input_variables']
)
replanner = replan_prompt | ChatOpenAI(model='gpt-4o-mini', temperature=0).with_structured_output(Act)

In [43]:
def replan_step(state: MyAgentState):
    output = replanner.invoke({'rewritten_query': state['rewritten_query'],
                               'plan': state['plan'],
                               'past_steps': state['past_steps']})
    

    if isinstance(output.action, Response):
        # Response를 반환하면 plan을 비우고 response 설정
        return {"response": output.action.response, "plan": []}
    

    print(output)
    
    next_plan = output.action.steps
    if len(next_plan) == 0:
        # plan이 비어있으면 완료
        return {"response": "No more steps needed.", "plan": []}
    else:
        return {"plan": next_plan}

In [44]:
def should_continue_replan(state: MyAgentState) -> Literal['final_report', 'execute']:
    plan = state.get("plan", [])
    
    if len(plan) > 0:
        return "execute"
    
    # if state["response"] == '':
    #     return "final_report"
    
    return "final_report"

### 2-3. Execute

In [45]:
rag_tool = create_rag_tool()
text2sql_app = text2sql_workflow(checkpointer=inmemory_saver)

In [46]:
@tool
def query_my_scores(query: str, runtime: ToolRuntime) -> str:
    """
    Converts the user's natural language query into SQL, retrieves the data from the database, and generates a natural language answer.

    Args:
        query: The user's natural language question.
        user_id: The user's ID.

    When to use:
    - When the user wants to view their exercise analysis data.
    - When querying for scores, trends, statistics, or other information stored in the database.
    - Examples: "What is the upper body score of my latest impression video?", "What are the pelvic area score trends for pommel horse videos?"

    Returns:
        The final answer or an error message.
    """
    config = runtime.config
    thread_id = config['configurable']['thread_id']
    user_id = config['configurable']['user_id']

    sub_config = {
        "configurable": {
            "thread_id": thread_id,
            "user_id": user_id
        }
    }
    
    try:
        # state 초기화: Annotated[List[str], add] 필드들은 반드시 리스트여야 함
        # checkpointer에서 이전 state를 불러올 때 타입이 맞지 않을 수 있으므로 명시적으로 초기화
        initial_state = {
            "origin_query": query,
            "rewritten_query": [],  # Annotated[List[str], add]이므로 리스트여야 함
            "errors": [],  # Annotated[List[str], add]이므로 리스트여야 함
            "sql": [],  # Annotated[List[str], add]이므로 리스트여야 함
            "selected_data": [],
            "final_answer": "",
            "cancelled": False,
            "error_num": 0,
            "direction": "",
            "only_data": True
        }
        
        result = text2sql_app.invoke(initial_state, config=sub_config)
        
        return result.get("final_answer", "No answer generated")
    
    except Exception as e:
        return f"Error executing query: {str(e)}"

In [47]:
sub_config = {
        "configurable": {
            "thread_id": '1',
            "user_id": '1'
        }
    }

query = "내가 가장 최근에 올린 용상 영상의 평균 점수를 조회해줘."
initial_state = {
            "origin_query": query,
            "rewritten_query": [],  # Annotated[List[str], add]이므로 리스트여야 함
            "errors": [],  # Annotated[List[str], add]이므로 리스트여야 함
            "sql": [],  # Annotated[List[str], add]이므로 리스트여야 함
            "selected_data": [],
            "final_answer": "",
            "cancelled": False,
            "error_num": 0,
            "direction": "",
            "only_data": True
        }


for chunk in text2sql_app.stream(initial_state, sub_config):
    print(chunk)


[NODE] synonum_process
[INPUT] origin_query: 내가 가장 최근에 올린 용상 영상의 평균 점수를 조회해줘.
[OUTPUT] rewritten_query: 내가 가장 최근에 올린 용상 영상의 평균 점수를 조회해줘.
{'synonum_process': {'rewritten_query': ['내가 가장 최근에 올린 용상 영상의 평균 점수를 조회해줘.']}}

[NODE] query_rewrite_sql
[INPUT] user query: 내가 가장 최근에 올린 용상 영상의 평균 점수를 조회해줘.
[OUTPUT] rewritten_query: 2026-01-02에 내가 가장 최근에 올린 용상 영상의 평균 점수를 조회.
{'sql_query_rewrite': {'rewritten_query': ['2026-01-02에 내가 가장 최근에 올린 용상 영상의 평균 점수를 조회.']}}

[NODE] query_gen_node
[INPUT] user_id: 1
[INPUT] user_question: 2026-01-02에 내가 가장 최근에 올린 용상 영상의 평균 점수를 조회.
[INPUT] errors count: 2
[INPUT] errors: ["Error: 1054 (42S22): Unknown column 'sport_agent.sp.name' in 'where clause'", 'Missing required security filter: pose_evaluation_sessions.user_id = 1 (alias allowed: e.g., ses.user_id = 1).']
[OUTPUT] Generated SQL:
SELECT 
  AVG(ps.user_score) AS average_score
FROM pose_scores AS ps
JOIN pose_evaluation_sessions AS ses ON ps.session_id = ses.id
JOIN sports AS sp ON ses.sport_id = sp.id
JOIN

In [48]:
execute_agent = create_agent(
    model=ChatOpenAI(model="gpt-4o-mini", temperature=0),
    tools=[query_my_scores, rag_tool],
    system_prompt="""You are an execution agent responsible for executing individual steps from a plan.

[Your role]
- Execute the specific task assigned to you from the plan
- Use the appropriate tools to complete the task
- Return clear, concise results that can be used for the next step

[Available tools]
1. rag_search: Search the 'Weightlifting Coaching Manual' document for information about techniques, training methods, scientific principles, etc.
   - Use when: You need to find information from coaching manuals or documents
   - Example: "Search for information about hip movement in Clean and Jerk"

2. query_my_scores: Query the user's exercise analysis data (scores, trends, statistics) from the database
   - Use when: The user asks about their own exercise data, scores, or performance trends
   - Example: "What is my latest score for upper body in Clean and Jerk?"

[Guidelines]
- Execute ONLY the task assigned to you in the current step
- Do not skip ahead or execute multiple steps at once
- Use tools when necessary to gather information
- Provide clear, factual answers based on tool results
- If a tool returns no results, state that clearly
- Keep your response focused on completing the assigned task

Your response will be used as input for the next step in the plan, so be precise and complete."""
)

In [49]:
def execute_step(state: MyAgentState, config: RunnableConfig):
    plan = state["plan"]
    
    plan_str = "\n".join(f"{i+1}. {step}" for i, step in enumerate(plan))
    task = plan[0]
    
    task_formatted = f"""For the following plan:
        {plan_str}\n\nYou are tasked with executing [step 1. {task}]."""
    
    configurable = config.get("configurable", {})
    thread_id = configurable.get("thread_id")
    user_id = configurable.get("user_id")

    sub_config = {
        "configurable": {
            "thread_id": thread_id,
            "user_id": user_id
        }
    }

    try:
        # 에이전트 실행 (내부적으로 query_my_scores tool 호출)
        agent_response = execute_agent.invoke(
            {"messages": [("user", task_formatted)]},
            config=sub_config
        )
        
        if "__interrupt__" in agent_response:
            interrupt_data = agent_response["__interrupt__"]
            
            # 사용자 입력 받기 (UI에서 처리됨)
            # interrupted_payload에는 rewritten_query와 user_data가 있음
            
            # 최상위 그래프도 human in the loop 필요하면 여기서 interrupt 발생
            user_decision = interrupt({
                "message": "Sub-graph requires human review",
                "rewritten_query": interrupt_data[0].value.get("rewritten_query"),
                "user_data": interrupt_data[0].value.get("user_data"),
            })
            
            resume_result = execute_agent.invoke(
                Command(
                    resume={
                        "cancelled": user_decision.get("cancelled", False),
                        "selected_data": user_decision.get("selected_data", []),
                        "user_input": user_decision.get("user_input", ""),
                    }
                ),
                config=sub_config
            )
            
            final_answer = resume_result["messages"][-1].content if resume_result.get("messages") else "No answer"
            
            return {
                "past_steps": [(task, final_answer)],
            }
        
        # Interrupt가 없으면 정상 결과 처리
        final_answer = agent_response["messages"][-1].content
        return {
            "past_steps": [(task, final_answer)],
        }
    
    except Exception as e:
        return {
            "past_steps": [(task, f"Error: {str(e)}")],
        }

### 2-4. Final Report
여기에 장기 메모리를 추가해야 함.

In [50]:
final_report_prompt = PromptTemplate.from_template(
    """You are given the objective and the previously done steps. Your task is to generate a final report in markdown format.
Final report should be written in professional tone.

Your objective was this:

{input}

Your previously done steps(question and answer pairs):

{past_steps}

Generate a final report in markdown format. Write your response in Korean."""
)

final_report_chain = (
    final_report_prompt
    | ChatOpenAI(model='gpt-4o-mini', temperature=0)
    | StrOutputParser()
)

In [51]:
def final_report(state: MyAgentState):
    past_steps = "\n\n".join(
        [
            f"Question: {past_step[0]}\n\nAnswer: {past_step[1]}\n\n####"
            for past_step in state["past_steps"]
        ]
    )

    response = final_report_chain.invoke({
        "input": state["rewritten_query"],
        "past_steps": past_steps
    })
    return {"response": response}

### 2-5. Plan and Execute Graph

In [52]:
planAndExecute_graph = StateGraph(MyAgentState)
planAndExecute_graph.add_node('plan', plan_step)
planAndExecute_graph.add_node('replan', replan_step)
planAndExecute_graph.add_node('execute', execute_step)
planAndExecute_graph.add_node('final_report', final_report)

planAndExecute_graph.add_edge(START, 'plan')
planAndExecute_graph.add_edge('plan', 'execute')
planAndExecute_graph.add_edge('execute', 'replan')
planAndExecute_graph.add_conditional_edges(
    'replan',
    should_continue_replan,
    {'final_report': 'final_report', 'execute': 'execute'}
)
planAndExecute_graph.add_edge('final_report', END)

planAndExecute_app = planAndExecute_graph.compile(checkpointer=inmemory_saver)

In [53]:
# planAndExecute 함수는 더 이상 필요 없음
# planAndExecute_app을 직접 노드로 사용
# def planAndExecute(state: MyAgentState, config: RunnableConfig):
#     configurable = config.get("configurable", {})
#     thread_id = configurable.get("thread_id")
#     user_id = configurable.get("user_id")
#     
#     inputs = {'rewritten_query': state['rewritten_query']}
#     
#     sub_config = {
#         "configurable": {
#             "thread_id": thread_id,
#             "user_id": user_id
#         }
#     }
#     
#     result = planAndExecute_app.invoke(
#         inputs,
#         config=sub_config
#     )
#     return {"response": result['response']}

## 3. SIMPLE node

In [54]:
simple_agent = create_agent(
    model=ChatOpenAI(model="gpt-4o-mini", temperature=0),
    tools=[query_my_scores, rag_tool],
    system_prompt="""You are a helpful agent.

[Your role]
- Execute the specific task assigned to you 
- Use the appropriate tools to complete the task

[Available tools]
1. rag_search: Search the 'Weightlifting Coaching Manual' document for information about techniques, training methods, scientific principles, etc.
   - Use when: You need to find information from coaching manuals or documents
   - Example: "Search for information about hip movement in Clean and Jerk"

2. query_my_scores: Query the user's exercise analysis data (scores, trends, statistics) from the database
   - Use when: The user asks about their own exercise data, scores, or performance trends
   - Example: "What is my latest score for upper body in Clean and Jerk?"

[Guidelines]
- Execute ONLY the task assigned to you
- Use tools when necessary to gather information
- Provide clear, factual answers based on tool results
- If a tool returns no results, state that clearly
- Keep your response focused on completing the assigned task
"""
)

In [55]:
def simpleAgent(state: MyAgentState, config: RunnableConfig):
    configurable = config.get("configurable", {})
    thread_id = configurable.get("thread_id")
    user_id = configurable.get("user_id")

    sub_config = {
        "configurable": {
            "thread_id": thread_id,
            "user_id": user_id
        }
    }
    
    try:
        # 에이전트 실행 (내부적으로 tool 호출 가능)
        result = simple_agent.invoke(
            {"messages": [("user", state['rewritten_query'])]},
            config=sub_config
        )
        
        # create_agent는 {"messages": [...]} 형식을 반환
        # interrupt 처리 (human-in-the-loop)
        if isinstance(result, dict) and "__interrupt__" in result:
            interrupt_data = result["__interrupt__"]
            
            # 최상위 그래프에서 human in the loop 처리
            user_decision = interrupt({
                "message": "Simple agent requires human review",
                "rewritten_query": interrupt_data[0].value.get("rewritten_query"),
                "user_data": interrupt_data[0].value.get("user_data"),
            })
            
            resume_result = simple_agent.invoke(
                Command(
                    resume={
                        "cancelled": user_decision.get("cancelled", False),
                        "selected_data": user_decision.get("selected_data", []),
                        "user_input": user_decision.get("user_input", ""),
                    }
                ),
                config=sub_config
            )
            
            # resume_result도 {"messages": [...]} 형식
            if isinstance(resume_result, dict) and resume_result.get("messages"):
                final_response = resume_result["messages"][-1].content
            else:
                final_response = "No answer"
            return {"response": str(final_response)}
        
        # 정상 결과 처리 (마지막 메시지가 최종 응답)
        if isinstance(result, dict) and result.get("messages"):
            final_response = result["messages"][-1].content
        else:
            # result가 다른 형식일 경우 처리
            final_response = str(result) if result else "No answer"
        
        # 문자열로 변환하여 반환 (state의 response 필드는 str 타입)
        return {"response": str(final_response)}
    
    except Exception as e:
        return {"response": f"Error: {str(e)}"}

## 4. Inapproprate node

In [56]:
def inappropriate(state: MyAgentState):
    user_input = state["input"]
    return {"response": f"your request ('{user_input}') is inappropriate for system."}

## 5. 전체 Graph

In [57]:
graph = StateGraph(MyAgentState)
graph.add_node('intentAnalyze', intentAnalyze)
graph.add_node('simple_node', simpleAgent)
graph.add_node('complex_node', planAndExecute_app)  # planAndExecute_app을 직접 노드로 추가
graph.add_node('inappropriate_node', inappropriate)

graph.add_conditional_edges(
    'intentAnalyze',
    should_continue_intent,
    {'SIMPLE': 'simple_node',
     'COMPLEX': 'complex_node',
     'INAPPROPRIATE': 'inappropriate_node'}
)

graph.add_edge(START, 'intentAnalyze')
graph.add_edge('simple_node', END)
graph.add_edge('complex_node', END)
graph.add_edge('inappropriate_node', END)

total_app = graph.compile(checkpointer=inmemory_saver)

In [58]:
def format_stream_output(chunk):
    """
    workflow.stream() 출력을 정돈된 형태로 변환 (기존 로직 유지 + replan 부분 버그 수정)
    replan 노드에서 'response' 필드가 있을 때, 내용을 상세히 출력하도록 개선.
    """
    path, data = chunk

    # 1. Path formatting
    if path:
        path_nodes = [p.split(':')[0] if ':' in p else p for p in path]
        path_str = " → ".join(path_nodes)
        last_node = path_nodes[-1]
        is_subgraph = len(path) >= 2
    else:
        # path가 빈 튜플이면 그래프 완료 후 최종 상태
        path_str = "FINAL"
        last_node = None
        path_nodes = []
        is_subgraph = False

    output_lines = [f"\n[{path_str}]", "-" * 60]

    # 2. 데이터가 비어있는 경우
    if not data:
        if is_subgraph and last_node == 'tools':
            output_lines.append("   🔧 Tool execution in progress...")
        else:
            output_lines.append("   (No data)")
        output_lines.append("")
        return "\n".join(output_lines)

    # 3. 각 노드별 출력 정리
    for node_name, node_data in data.items():
        if node_name == 'intentAnalyze':
            output_lines.append(f"📋 Intent Analysis:")
            output_lines.append(f"   Category: {node_data.get('intent_category', 'N/A')}")
            output_lines.append(f"   Rewritten Query: {node_data.get('rewritten_query', 'N/A')}")

        elif node_name == 'plan':
            output_lines.append(f"📝 Plan:")
            plan_steps = node_data.get('plan', [])
            for i, step in enumerate(plan_steps, 1):
                output_lines.append(f"   {i}. {step}")

        elif node_name == 'execute':
            output_lines.append(f"⚙️  Execute:")
            past_steps = node_data.get('past_steps', [])
            for step_num, (task, result) in enumerate(past_steps, 1):
                output_lines.append(f"   Step {step_num}: {task}")
                # 결과가 너무 길면 잘라서 표시
                result_preview = result
                output_lines.append(f"   Result: {result_preview}")

        elif node_name == 'replan':
            output_lines.append(f"🔄 Re-plan:")
            # plan이 있으면 step 단위로 표시
            if 'plan' in node_data:
                plan_steps = node_data.get('plan', [])
                for i, step in enumerate(plan_steps, 1):
                    output_lines.append(f"   {i}. {step}")
            # response가 있으면 해당 내용을 상세히 출력 (기존 로직에서는 단순 메세지여서 결과가 안 보임)
            if 'response' in node_data:
                final_response = node_data.get('response')
                if isinstance(final_response, dict):
                    # dict라면 key:value 표시
                    for k, v in final_response.items():
                        output_lines.append(f"   {k}: {v}")
                elif isinstance(final_response, list):
                    for i, v in enumerate(final_response, 1):
                        output_lines.append(f"   {i}. {v}")
                else:
                    output_lines.append(f"   → {final_response}")

        elif node_name == 'final_report':
            output_lines.append(f"📄 Final Report:")
            response = node_data.get('response', 'N/A')
            output_lines.append(f"   {response}")

        elif node_name == 'response':
            output_lines.append(f"💬 Response:")
            response = node_data if isinstance(node_data, str) else node_data.get('response', 'N/A')
            output_lines.append(f"   {response}")

        elif node_name == 'simple_node':
            output_lines.append(f"💬 Simple Agent Response:")
            response = node_data.get('response', 'N/A')
            output_lines.append(f"   {response}")

        elif node_name == 'complex_node':
            output_lines.append(f"💬 Complex Agent Response:")
            response = node_data.get('response', 'N/A')
            output_lines.append(f"   {response}")

        elif node_name == 'inappropriate_node':
            output_lines.append(f"💬 Inappropriate Response:")
            response = node_data.get('response', 'N/A')
            output_lines.append(f"   {response}")

        elif node_name == 'model':
            messages = node_data.get('messages', [])
            if messages:
                last_msg = messages[-1]
                if hasattr(last_msg, 'tool_calls') and getattr(last_msg, 'tool_calls'):
                    tool_name = last_msg.tool_calls[0].get('name', 'unknown')
                    output_lines.append(f"🔧 Tool Call: {tool_name}")
                elif hasattr(last_msg, 'content') and last_msg.content:
                    content_preview = last_msg.content
                    output_lines.append(f"💭 Model Output: {content_preview}")

        elif node_name == 'tools':
            messages = node_data.get('messages', [])
            if messages:
                last_msg = messages[-1]
                if hasattr(last_msg, 'content'):
                    content_preview = str(last_msg.content)
                    output_lines.append(f"✅ Tool Result: {content_preview}")

    # subgraph 내부 노드 (특히 tools) 처리
    if is_subgraph and last_node == 'tools':
        found_tool_info = False
        for key, value in data.items():
            if isinstance(value, dict):
                if 'messages' in value:
                    messages = value.get('messages', [])
                    if messages:
                        last_msg = messages[-1]
                        # ToolMessage인 경우
                        if hasattr(last_msg, 'name') and hasattr(last_msg, 'content'):
                            tool_name = getattr(last_msg, 'name', 'unknown')
                            content = str(last_msg.content)
                            content_preview = content
                            output_lines.append(f"✅ Tool Result ({tool_name}): {content_preview}")
                            found_tool_info = True
                        elif hasattr(last_msg, 'content') and last_msg.content:
                            content_preview = str(last_msg.content)
                            output_lines.append(f"💭 Tool Output: {content_preview}")
                            found_tool_info = True
                elif key in ['tool_calls', 'tool_results']:
                    value_str = str(value)
                    output_lines.append(f"✅ Tool Result: {value_str}")
                    found_tool_info = True
            elif key not in ['model', 'tools', 'intentAnalyze', 'plan', 'execute', 'replan', 'final_report', 'response']:
                if isinstance(value, dict):
                    value_str = str(value)
                    output_lines.append(f"🔧 Tool Execution ({key}): {value_str}")
                    found_tool_info = True
        if not found_tool_info and data:
            first_key = list(data.keys())[0]
            first_value = data[first_key]
            if isinstance(first_value, dict):
                if 'messages' in first_value:
                    messages = first_value['messages']
                    if messages:
                        last_msg = messages[-1]
                        if hasattr(last_msg, 'content'):
                            content_preview = str(last_msg.content)
                            output_lines.append(f"🔧 Tool Execution: {content_preview}")
                        else:
                            output_lines.append(f"🔧 Tool Execution: {str(last_msg)}")
                else:
                    output_lines.append(f"🔧 Tool Execution ({first_key}): {str(first_value)}")
            else:
                output_lines.append(f"🔧 Tool Execution: {str(first_value)}")

    # 아무 출력도 없었고 data가 비어있지 않다면 raw data 표시
    if len(output_lines) == 2:  # path와 구분선만 있는 경우
        if is_subgraph:
            output_lines.append(f"   (Subgraph node: {last_node})")
        output_lines.append(f"   (Data keys: {list(data.keys())})")

    output_lines.append("")
    return "\n".join(output_lines)

In [59]:
inputs = MyAgentState({
    'input': '가장 최근에 올린 영상분석 결과를 조회해줘.',
    'chat_history': [],
    'response': ''
})

config=RunnableConfig(
    recursion_limit=30,
    configurable={"thread_id": '1',
                  "user_id": '1'}
    )

for chunk in total_app.stream(
    inputs, 
    subgraphs=True, 
    config=config,
    stream_mode='updates'
    ):
    # print(chunk, '\n')
    print(format_stream_output(chunk))


[FINAL]
------------------------------------------------------------
📋 Intent Analysis:
   Category: SIMPLE
   Rewritten Query: Retrieve the analysis results of the most recent video uploaded by the user.


[FINAL]
------------------------------------------------------------
💬 Simple Agent Response:
   It seems there was an error while trying to retrieve the analysis results of your most recent video. Please try again later or check if there are any specific details you can provide to assist in the query.



In [57]:
chunk

((),
 {'simple_node': {'response': 'Retrieve the analysis results of the most recently uploaded video.'}})

In [60]:
inputs = MyAgentState({
    'input': '나의 clean and jerk 동작에서 골반과 관련해서 문제점을 파악하고 해결방법을 간략하게 알려줘.',
    'chat_history': [],
    'response': ''
})

config=RunnableConfig(
    recursion_limit=30,
    configurable={"thread_id": '1',
                  "user_id": '1'}
    )

# result = total_app.invoke(inputs, config=config)


for chunk in total_app.stream(
    inputs, 
    subgraphs=True, 
    config=config,
    stream_mode='updates'
    ):
    # print(chunk, '\n')
    print(format_stream_output(chunk))


[FINAL]
------------------------------------------------------------
📋 Intent Analysis:
   Category: COMPLEX
   Rewritten Query: Analyze the user's Clean and Jerk technique to identify issues related to the pelvis and provide a brief summary of potential solutions.


[complex_node]
------------------------------------------------------------
📝 Plan:
   1. Search the 'Weightlifting Coaching Manual' document for information on Clean and Jerk technique, specifically focusing on pelvis positioning and common issues.
   2. Analyze the retrieved information to identify key issues related to the pelvis during the Clean and Jerk.
   3. Summarize the identified issues and potential solutions based on the analysis of the Clean and Jerk technique.
   4. Generate final answer with a brief summary of the pelvis-related issues and potential solutions for the user's Clean and Jerk technique.


[complex_node]
------------------------------------------------------------
⚙️  Execute:
   Step 1: Search 

GraphRecursionError: Recursion limit of 30 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT

In [61]:
inputs = MyAgentState({
    'input': '나의 clean and jerk 동작에서 골반과 관련해서 문제점을 파악하고 해결방법을 간략하게 알려줘.',
    'chat_history': [],
})

config=RunnableConfig(
    recursion_limit=30,
    configurable={"thread_id": '2',
                  "user_id": '1'}
    )

for chunk in total_app.stream(
    inputs, 
    subgraphs=True, 
    config=config,
    stream_mode='updates'
    ):
    print(format_stream_output(chunk))


[FINAL]
------------------------------------------------------------
📋 Intent Analysis:
   Category: COMPLEX
   Rewritten Query: Analyze the user's Clean and Jerk performance to identify issues related to the pelvis movement and provide concise solutions for improvement.


[complex_node]
------------------------------------------------------------
📝 Plan:
   1. Search the database for the user's past performance scores related to the Clean and Jerk movement.
   2. Analyze the retrieved performance scores to identify any patterns or issues specifically related to pelvis movement during the Clean and Jerk.
   3. Consult the 'Weightlifting Coaching Manual' to find information on common issues with pelvis movement in the Clean and Jerk and recommended solutions for improvement.
   4. Compile the identified issues and corresponding solutions into a concise format for the user.
   5. Generate final answer with the analysis and solutions for the user's Clean and Jerk performance.


[complex_

GraphRecursionError: Recursion limit of 30 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT

In [ ]:
chunk[1]['complex_node']['response']

"# 최종 보고서\n\n## 목표\nClean and Jerk 동작에서 골반과 관련된 문제를 식별하고 간단한 해결책을 제시하는 것입니다.\n\n## 주요 발견\n'Weightlifting Coaching Manual' 문서에서 Clean and Jerk 동작에서의 일반적인 골반 문제에 대한 정보를 찾았습니다. 다음은 주요 내용입니다:\n\n1. **기술 구조**: Clean and Jerk는 두 가지 주요 동작으로 구성됩니다: Clean(바벨을 가슴까지 들어올리는 동작)과 Jerk(바벨을 머리 위로 들어올리는 동작). 각 동작은 여러 단계와 요소로 나뉘며, 이는 다양한 신체 위치와 움직임을 포함합니다.\n\n2. **골반 위치**: 리프트 중 엉덩이의 높이는 성능에 큰 영향을 미칠 수 있습니다. 낮은 엉덩이 위치는 하체 힘을 더 잘 활용할 수 있게 해주지만, 높은 엉덩이 위치는 리프트의 비효율성을 초래할 수 있습니다.\n\n3. **일반적인 문제**:\n   - **엉덩이 높이**: 엉덩이 위치가 높은 선수는 Clean에서 어려움을 겪을 수 있으며, 이는 잘못된 리프팅 메커니즘과 허리에 대한 스트레스를 증가시킬 수 있습니다.\n   - **무릎 및 엉덩이 각도**: Jerk 동안 무릎과 엉덩이의 각도는 매우 중요합니다. Jerk Dip 동안 적절한 무릎 각도(약 100-110도)는 효과적인 힘 생성에 필요합니다.\n\n4. **훈련 고려사항**: 매뉴얼은 선수의 신체 비율과 유연성에 따라 개별화된 훈련의 중요성을 강조합니다. 각 선수에 맞게 기술 조정이 필요할 수 있습니다.\n\n## 결론\n이 정보는 사용자의 성과 점수를 분석하고 Clean and Jerk 기술과 관련된 특정 골반 문제를 식별하는 데 유용할 것입니다. 각 선수의 특성에 맞춘 훈련과 기술 조정이 필요하며, 이를 통해 성능을 최적화할 수 있습니다."

In [ ]:
inputs = MyAgentState({
    'input': '다른 사용자가 가장 최근에 올린 영상에서 문제가 되는 신체부위를 알려줘.',
    'chat_history': [],
})

config=RunnableConfig(
    recursion_limit=15,
    configurable={"thread_id": '1',
                  "user_id": '1'}
    )

for chunk in total_app.stream(inputs, subgraphs=True, config=config):
    print(format_stream_output(chunk))


[START]
------------------------------------------------------------
📋 Intent Analysis:
   Category: COMPLEX
   Rewritten Query: Identify the problematic body parts in the most recent video uploaded by another user.


[complex_node]
------------------------------------------------------------
📝 Plan:
   1. Search the document for guidelines on identifying problematic body parts in weightlifting videos.
   2. Retrieve the most recent video uploaded by the user from the database.
   3. Analyze the video based on the guidelines found in the document to identify any problematic body parts.
   4. Compile the findings into a clear report detailing the identified problematic body parts.
   5. Generate final answer with the report on the identified problematic body parts.


[complex_node]
------------------------------------------------------------
⚙️  Execute:
   Step 1: Search the document for guidelines on identifying problematic body parts in weightlifting videos.
   Result: I found sever

In [40]:
chunk

((),
 {'complex_node': {'response': '# 최종 보고서\n\n## 목적\n최근 다른 사용자가 업로드한 비디오에서 문제 있는 신체 부위를 식별하는 것입니다.\n\n## 분석 단계\n비디오 분석을 위한 가이드라인을 문서에서 검색한 결과, 다음과 같은 주요 포인트를 추출하였습니다:\n\n1. **시작 자세 및 리프트 메커니즘**:\n   - 성공적인 리프트를 위해 시작 자세가 중요합니다. 이는 상체를 곧게 유지하고 다리 힘을 사용하여 리프트를 시작하는 것을 포함합니다.\n   - 리프트 중 바벨이 떨어지지 않도록 팔꿈치를 높게 유지해야 합니다.\n\n2. **일반적인 오류**:\n   - 많은 운동선수들이 리프트 중 균형을 유지하는 데 어려움을 겪으며, 종종 너무 앞으로 또는 뒤로 기울어져 리프트에 실패할 수 있습니다.\n   - 발을 단단히 고정하고 주요 근육(대퇴사두근 및 승모근)을 효과적으로 사용해야 합니다.\n\n3. **신체 유형 고려사항**:\n   - 운동선수는 신체 유형(예: 긴 팔다리 vs. 짧은 팔다리)에 따라 분류되며, 이는 리프팅 기술에 영향을 미칠 수 있습니다. 이러한 차이를 이해하는 것은 성능에서 잠재적인 문제를 식별하는 데 중요합니다.\n\n4. **특정 기술**:\n   - 문서에서는 바벨을 몸 가까이에 유지하고 리프트 중 무릎이 과도하게 앞으로 움직이지 않도록 하는 것의 중요성과 같은 특정 리프팅 기술을 설명합니다.\n\n5. **훈련 권장 사항**:\n   - 개인의 신체적 특성과 관절 유연성을 고려한 체계적인 훈련 프로그램의 필요성을 강조하여 부상을 예방하고 성능을 향상시킬 수 있습니다.\n\n이러한 가이드라인은 사용자의 웨이트리프팅 비디오를 분석하여 문제 있는 신체 부위를 식별하는 데 유용할 것입니다. \n\n## 결론\n위의 분석을 바탕으로, 비디오에서 문제 있는 신체 부위를 효과적으로 식별하고 개선할 수 있는 방법을 제시할 수 있습니다. 이를 통해 운동선수의 성능 향상과 부상 예방에 기여할 수 있을 

# 최종 보고서

## 분석 목표
이번 분석의 목표는 클린 앤 저크 동작에 대한 모든 결과를 바탕으로 골반과 관련된 트렌드를 분석하고, 문제가 발견될 경우 해결책을 제시하는 것이었습니다.

## 성과 점수 요약
사용자의 클린 앤 저크 동작에 대한 과거 성과 점수는 다음과 같습니다:

### 점수 상태별 요약
1. **상태별 점수:**
   - Connection 11-12: 85.7200 (경고)
   - Connection 11-13: 83.9000 (기준 이하)
   - Connection 11-23: 91.0500 (기준 이하)
   - Connection 12-14: 82.3700 (경고)
   - Connection 12-24: 91.5000 (경고)
   - Connection 13-15: 83.2400 (경고)
   - Connection 14-16: 81.4100 (기준 이하)
   - Connection 23-24: 86.5900 (경고)
   - Connection 23-25: 88.3800 (경고)
   - Connection 24-26: 86.0600 (기준 이하)
   - Connection 25-27: 91.2200 (기준 이하)
   - Connection 26-28: 90.9300 (기준 이하)
   - 전체: 85.2000 (기준 이하)

2. **달성 상태별 점수:**
   - Connection 11-12: 89.1500 (달성)
   - Connection 11-13: 84.2200 (경고)
   - Connection 11-23: 94.1700 (경고)
   - Connection 12-14: 84.2800 (경고)
   - Connection 12-24: 94.9100 (경고)
   - Connection 13-15: 85.3600 (경고)
   - Connection 14-16: 83.2300 (경고)
   - Connection 23-24: 92.3600 (달성)
   - Connection 23-25: 91.7900 (경고)
   - Connection 24-26: 89.7900 (경고)
   - Connection 25-27: 94.4300 (경고)
   - Connection 26-28: 94.7800 (경고)
   - 전체: 88.4000 (경고)

3. **최고 달성 상태별 점수:**
   - Connection 11-12: 96.4200 (달성)
   - Connection 11-13: 96.7800 (달성)
   - Connection 11-23: 98.9800 (달성)
   - Connection 12-14: 95.4100 (달성)
   - Connection 12-24: 99.0800 (달성)
   - Connection 13-15: 96.0500 (달성)
   - Connection 14-16: 93.8400 (달성)
   - Connection 23-24: 96.9600 (달성)
   - Connection 23-25: 98.1600 (달성)
   - Connection 24-26: 97.5400 (달성)
   - Connection 25-27: 98.9500 (달성)
   - Connection 26-28: 99.0300 (달성)
   - 전체: 95.9200 (달성)

## 분석 결과
클린 앤 저크 동작에 대한 성과 점수 분석 결과, 다음과 같은 트렌드와 문제가 발견되었습니다:

1. **왼쪽 엉덩이와 왼쪽 무릎 연결:**
   - 점수: 88.38 (경고), 91.79 (경고), 98.16 (달성)
   - 상태: 두 점수가 경고 범주에 있으며, 마지막 점수는 만족스러운 수준에 도달했습니다.

2. **오른쪽 엉덩이와 오른쪽 무릎 연결:**
   - 점수: 86.06 (기준 이하), 89.79 (경고), 97.54 (달성)
   - 상태: 첫 번째 점수는 기준 이하로, 심각한 문제가 있음을 나타내며, 두 번째 점수는 경고 범주에 있고, 마지막 점수는 만족스러운 수준에 도달했습니다.

### 문제 요약
- 오른쪽 엉덩이와 오른쪽 무릎 연결에 대해 지속적인 경고와 하나의 기준 이하 점수가 있어, 클린 앤 저크 동작 중 오른쪽 측면의 자세에 주의가 필요합니다.
- 왼쪽 측면은 개선이 있지만 여전히 경고가 있어 해결이 필요합니다.

## 해결책 제안
'웨이트리프팅 코칭 매뉴얼'을 참고하여 골반 위치와 관련된 해결책 및 교정 기술을 제안합니다:

1. **골반 위치 및 발 위치:**
   - 클린 앤 저크 동작 중 발 위치는 일반적으로 바벨의 수직선과 일치해야 하며, 발은 어깨 너비로 벌려야 합니다. 바벨은 클린 시 중발가락 관절에서 1-3cm 뒤에 위치해야 하며, 이는 체형에 따라 달라질 수 있습니다.
   - 발과 지면 간의 강한 연결을 유지하여 효과적인 리프팅 메커니즘을 보장해야 합니다.

2. **엉덩이와 무릎 연결을 위한 교정 기술:**
   - 오른쪽 엉덩이와 오른쪽 무릎 간의 강하고 안정적인 연결을 유지하는 것이 중요합니다. 이는 무릎이 안쪽으로 무너지지 않도록 하고, 엉덩이가 동작 내내 활성화되도록 함으로써 달성할 수 있습니다.
   - 퀴드리셉스와 엉덩이 안정근을 강화하여 연결을 향상시키고 불안정성을 초래할 수 있는 측면 이동을 방지해야 합니다.

3. **왼쪽 측면 문제에 대한 경고:**
   - 선수들이 리프트 중 왼쪽 측면에 더 의존하는 경향이 있어 불균형이 발생할 수 있습니다. 이는 오른쪽 성능을 저해하지 않으면서 왼쪽 측면을 강화하는 단일 운동을 포함하여 해결해야 합니다.
   - 코치는 리프트 중 양쪽이 균등하게 참여하도록 선수의 자세를 면밀히 모니터링해야 합니다.

4. **훈련 추천:**
   - 골반의 올바른 위치와 엉덩이와 무릎 간의 연결을 강조하는 특정 드릴을 포함해야 합니다. 단일 다리 스쿼트 및 측면 밴드 걷기와 같은 운동이 올바른 메커니즘을 강화하는 데 도움이 될 수 있습니다.
   - 정기적인 피드백과 비디오 분석을 통해 선수들이 리프팅 기술의 편차를 인식하고 수정할 수 있도록 지원해야 합니다.

이 보고서는 클린 앤 저크 동작에서 골반과 관련된 문제를 해결하기 위한 기초 자료로 활용될 것입니다.

In [ ]:
# 나중에 추가 해야함.
user_decision = {
    "cancelled": input("취소? (y/n): ").lower() == 'y',
    "selected_data": [],  # 사용자 선택 데이터
    "user_input": input("수정된 쿼리: ") if input("쿼리 수정? (y/n): ").lower() == 'y' else ""
}

for chunk in total_app.stream(
    Command(resume=user_decision),
    subgraphs=True,
    config=config
):
    print(format_stream_output(chunk))

In [ ]:
inputs = MyAgentState({
    'input': '내가 올린 모든 clean and jerk 분석 결과에서 골반에 대한 동향을 알려줘. 그 다음 문제점이 있으면 해결방법을 제시해줘.',
    'chat_history': []
})

for chunk in total_app.stream(inputs, subgraphs=True):
    print(format_stream_output(chunk))


[START]
------------------------------------------------------------
📋 Intent Analysis:
   Category: COMPLEX
   Rewritten Query: Analyze the trends related to the pelvis in all my clean and jerk analysis results, and if any issues are identified, provide solutions.


[complex_node]
------------------------------------------------------------
📝 Plan:
   1. Search the database for the user's past performance scores related to the clean and jerk movement.
   2. Analyze the retrieved performance scores to identify any trends or issues related to the pelvis during the clean and jerk.
   3. If issues are identified, consult the 'Weightlifting Coaching Manual' for recommended solutions or corrective techniques for pelvis-related issues in the clean and jerk.
   4. Compile the analysis results and solutions into a clear report.
   5. Generate final answer with the analysis and solutions provided to the user.


[complex_node → execute]
----------------------------------------------------------

RemoteProtocolError: Server disconnected without sending a response.